In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, precision_score, f1_score

# 1. Load and Clean
df = pd.read_csv('creditcard.csv')
df = df.drop(['Time'], axis=1)

# 2. Scale Amount
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))

# 3. Split (Crucial: Use stratify)
X = df.drop(['Class'], axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Create the SMOTE data for the models to use
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Setup Complete! X_train_smote is now defined and ready for the tournament.")

c:\Users\My PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\My PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


Setup Complete! X_train_smote is now defined and ready for the tournament.


In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix
import pandas as pd

# We use the SMOTE data from Step 3 for all models to keep the test fair
# X_train_smote, y_train_smote, X_test, y_test are used here
print("Tournament bracket is set: Logistic Regression, Decision Tree, Random Forest, and XGBoost.")

Tournament bracket is set: Logistic Regression, Decision Tree, Random Forest, and XGBoost.


In [5]:
#Create a dictionary of models 
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1 ),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metrics='logloss', random_state=42)

}
#Dictonary to store results
results = []
for name, model in models.items():
    #Train the model 
    model.fit(X_train_smote, y_train_smote)
    #Make predictions
    y_pred = model.predict(X_test)
    #Calcuate metrics
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    #save our results list
    results.append({
        "Model": name,
        "Recall": round(recall,4),
        "precision": round(precision,4),
        "f1_score": round(f1,4)
    })
    #convert to a table
    results_df = pd.DataFrame(results).sort_values(by="Recall", ascending=False)
    print(results_df)


                Model  Recall  precision  f1_score
0  LogisticRegression  0.9184     0.0563    0.1061
                Model  Recall  precision  f1_score
0  LogisticRegression  0.9184     0.0563    0.1061
1       Decision Tree  0.7857     0.3598    0.4936
                Model  Recall  precision  f1_score
0  LogisticRegression  0.9184     0.0563    0.1061
2       Random Forest  0.8265     0.8710    0.8482
1       Decision Tree  0.7857     0.3598    0.4936


c:\Users\My PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:54:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "eval_metrics", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


                Model  Recall  precision  f1_score
0  LogisticRegression  0.9184     0.0563    0.1061
3             XGBoost  0.8673     0.6855    0.7658
2       Random Forest  0.8265     0.8710    0.8482
1       Decision Tree  0.7857     0.3598    0.4936
